# Day 02 — ELT and Incremental Silver

**Student:** Danah Almudaifer  
**Project:** Masar Mini-Lakehouse  
**Programme:** Modern Data Engineering for AI Systems (SDA-DSC-214)

Covers **LAB 03** (parts 3a and 3b).

This notebook preserves the executed evidence from my completed Colab run. The project uses only the supplied synthetic Masar dataset.


In [33]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))

Continue workspace: outputs/day01_bronze_lo_ncbvo


In [34]:
from masar.silver import run_staging_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_staging_lab(spark, SOURCE, WORK)
    validate_stage_result('lab03a_staging', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('Observed staging row counts:', result['counts'])
    preview = WORK / ('mini_lakehouse/staging/day02_' + result['run_id'] + '/stg_trips')
    spark.read.format('delta').load(str(preview)).select('trip_id', 'city', 'fare_sar').orderBy('trip_id').show(5, truncate=False)
finally:
    spark.stop()

{
  "scope": "DAY02_STAGING_ENGINE",
  "checks": {
    "counts_verified": true,
    "typed_values_match_source_oracle": true,
    "drivers_unique_and_join_safe": true,
    "gps_valid": true,
    "delta_readback": true
  }
}
Observed staging row counts: {'stg_trips': 144, 'stg_drivers': 6, 'stg_gps': 216}
+---------+------+--------+
|trip_id  |city  |fare_sar|
+---------+------+--------+
|SYN_T0001|Riyadh|18.00   |
|SYN_T0001|Riyadh|18.00   |
|SYN_T0002|Riyadh|19.25   |
|SYN_T0002|Riyadh|19.25   |
|SYN_T0003|Riyadh|20.50   |
+---------+------+--------+
only showing top 5 rows



In [35]:
from masar.silver import run_incremental_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_incremental_lab(spark, SOURCE, WORK)
    validate_stage_result('lab03b_silver', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    spark.read.format('delta').load(str(WORK/'mini_lakehouse/silver/trips')).select('trip_id', 'city', 'fare_sar').orderBy('trip_id').show(5, truncate=False)
finally:
    spark.stop()

{
  "scope": "DAY02_SILVER_ENGINE",
  "checks": {
    "actual_delta_files": true,
    "all_scenarios_match_independent_oracle": true,
    "business_keys_unique": true,
    "late_rows_retained": true,
    "replay_preserves_business_content": true
  }
}
+-----------+------+--------+
|trip_id    |city  |fare_sar|
+-----------+------+--------+
|SYN_LATE001|Riyadh|25.00   |
|SYN_LATE002|Jeddah|27.00   |
|SYN_LATE003|Dammam|29.00   |
|SYN_T0001  |Riyadh|18.00   |
|SYN_T0002  |Riyadh|19.25   |
+-----------+------+--------+
only showing top 5 rows



In [23]:
%pip install -q dbt-core==1.9.8 dbt-spark==1.9.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.4/114.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.0/947.0 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.9/144.9 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.0/167.0 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.7/442.7 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.2/92.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [36]:
from masar.dbt_lab import run_dbt_lab
dbt_report, dbt_path = run_dbt_lab(ROOT)
print(json.dumps({'status': dbt_report['status'], 'phases_completed': len(dbt_report['phases']), 'report': str(dbt_path.relative_to(ROOT)), 'error': dbt_report.get('error')}, indent=2))
assert dbt_report['status'] == 'PASSED_DBT_NATIVE', dbt_report.get('error')

# Observed learning output
for phase in dbt_report['phases']:
    print(phase['phase'], 'rows:', phase['rows'], 'fare SAR:', phase['total_fare_sar'])
print('Catalog evidence:', dbt_report['commands'][-1])

{
  "status": "PASSED_DBT_NATIVE",
  "phases_completed": 4,
  "report": "outputs/dbt_validation_q0bblj7d/reports/dbt_attempt.json",
  "error": null
}
base rows: 72 fare SAR: 1794.60
rerun rows: 72 fare SAR: 1794.60
late rows: 75 fare SAR: 1875.60
late_replay rows: 75 fare SAR: 1875.60
Catalog evidence: {'catalog_sha256': 'aa327f87dd99d4f78fc92150fda00dc601bb0929fcc626f8c75d40d55b0e5165', 'models_documented': 6, 'sources_documented': 3, 'metadata_method': 'native DESCRIBE TABLE EXTENDED', 'phase': 'documentation', 'command': ['docs', 'generate'], 'target': 'dbt/commands/documentation/target'}


In [37]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day02_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    dbt_workspace = dbt_path.parent.parent
    for p in sorted(dbt_workspace.rglob('*')):
        if p.is_file():
            bundle.write(p, p.relative_to(ROOT).as_posix())
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))

Retain the notebook outputs, notes and outputs/day02_handoff.zip


# Summary to understand flow pipeline


  
               SOURCE SYSTEM
                     │
                     │
          CSV (raw files)
                     │
                     ▼
           ┌─────────────────┐
           │     PySpark     │
           │ processing tool │
           └────────┬────────┘
                    │ writes
                    ▼
        ┌───────────────────────┐
        │     BRONZE DELTA      │
        │                       │
        │ Raw deliveries        │
        │ Preserve source       │
        │ Replay allowed        │
        └───────────┬───────────┘
                    │
              PySpark cleans
                    │
                    ▼
        ┌───────────────────────┐
        │       STAGING         │
        │                       │
        │ Correct data types    │
        │ Normalize fields      │
        │ Prepare for business  │
        │ logic                 │
        └───────────┬───────────┘
                    │
        PySpark/business rules
                    │
                    ▼
        ┌───────────────────────┐
        │      SILVER DELTA     │
        │                       │
        │ Deduplicated          │
        │ Clean                 │
        │ Incremental           │
        │ Business trustworthy  │
        └───────────┬───────────┘
                    │
                    │ dbt models
                    ▼
          ┌──────────────────┐
          │       dbt        │
          │                  │
          │ SQL models       │
          │ Tests            │
          │ Incremental logic│
          │ Documentation    │
          └────────┬─────────┘
                   │
                   ▼
        ┌─────────────────────┐
        │ Final data models   │
        │                     │
        │ Analytics           │
        │ Dashboards          │
        │ Reporting           │
        └─────────────────────┘